# CTC & sequence transduction — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def hz_to_mel(f): return 2595*np.log10(1+np.asarray(f)/700)
def mel_to_hz(m): return 700*(10**(np.asarray(m)/2595)-1)
def stft_mag(x,N=256,H=64):
    w=np.hanning(N); frames=[]
    for s in range(0,len(x)-N+1,H): frames.append(np.abs(np.fft.rfft(x[s:s+N]*w)))
    return np.array(frames).T
print('setup ok')

## Summing monotonic alignments with blanks
CTC trains without frame labels by letting many paths collapse to the same transcript. These examples show collapse, enumeration, forward recursion, repeated letters, and the loss.

In [ ]:
# 1) collapse repeats then remove blanks
blank=0; names={0:'_',1:'a',2:'b'}; path=[0,1,1,0,2,2,0]; collapsed=[]; prev=None
for s in path:
    if s!=prev: collapsed.append(s)
    prev=s
label=[s for s in collapsed if s!=blank]
plt.figure(figsize=(6,3)); plt.step(range(len(path)),path,where='mid'); plt.yticks([0,1,2],['_','a','b']); plt.title('path collapses to ab')
assert label==[1,2]

In [ ]:
# 2) enumerate all paths for target a on three frames
probs=np.array([[0.6,0.3,0.1],[0.5,0.4,0.1],[0.2,0.6,0.2]]); vals=[]
def collapse(path):
    out=[]; prev=None
    for s in path:
        if s!=prev: out.append(s)
        prev=s
    return tuple(s for s in out if s!=0)
for path in itertools.product(range(3), repeat=3): vals.append((path,collapse(path),np.prod([probs[t,s] for t,s in enumerate(path)])))
pa=sum(p for path,lab,p in vals if lab==(1,)); best=max([v for v in vals if v[1]==(1,)],key=lambda z:z[2])
plt.figure(figsize=(6,3)); plt.bar(['best path','sum paths'],[best[2],pa]); plt.title('CTC sums alignments')
assert abs(pa-0.498)<1e-9 and best[0]==(0,0,1)

In [ ]:
# 3) forward recursion matches enumeration for target a
probs=np.array([[0.6,0.3,0.1],[0.5,0.4,0.1],[0.2,0.6,0.2]]); alpha=np.array([1.0,0,0]); hist=[]
for row in probs:
    b,a,_=row; alpha=np.array([alpha[0]*b,(alpha[0]+alpha[1])*a,(alpha[1]+alpha[2])*b]); hist.append(alpha.copy())
plt.figure(figsize=(6,3)); plt.plot(np.array(hist),'-o'); plt.legend(['blank-start','a','blank-end']); plt.title('forward masses for target a')
assert np.allclose(np.round(alpha,4),[0.06,0.396,0.102]) and abs(alpha[1]+alpha[2]-0.498)<1e-9

In [ ]:
# 4) blanks allow repeated letters to survive collapse
paths=[[1,1,1],[1,0,1]]; labels=[collapse(p) for p in paths]
plt.figure(figsize=(6,3)); plt.imshow(np.array(paths),aspect='auto'); plt.yticks([0,1],['aaa','a_a']); plt.colorbar(ticks=[0,1]); plt.title('blank separates repeated a')
assert labels[0]==(1,) and labels[1]==(1,1)

In [ ]:
# 5) CTC loss falls when valid-path probability rises
pa=0.498; sharper=np.array([[0.8,0.15,0.05],[0.7,0.25,0.05],[0.1,0.85,0.05]]); vals=[]
for path in itertools.product(range(3), repeat=3): vals.append((collapse(path),np.prod([sharper[t,s] for t,s in enumerate(path)])))
p2=sum(p for lab,p in vals if lab==(1,)); losses=[-math.log(pa),-math.log(p2)]
plt.figure(figsize=(6,3)); plt.bar(['base','sharper'],losses); plt.ylabel('CTC loss'); plt.title('more valid mass -> lower loss')
assert losses[1]<losses[0]